# Wrapped Normal Mixture Model on the Poincaré Disk

In the previous notebook (`hyperbolic_kmeans.ipynb`), we clustered data on the Poincaré disk using **hard assignments** -- each point belonged to exactly one cluster, with centroids computed via the Einstein midpoint. That approach is the hyperbolic analogue of Euclidean K-means.

Now we move from hard to **soft** clustering using the **Expectation-Maximization (EM)** algorithm with a mixture of **wrapped normal distributions** on the Poincaré disk.

### Why Euclidean Gaussians Don't Work Here

The Poincaré disk has **negative curvature** -- distances grow exponentially near the boundary. A standard Euclidean Gaussian $\mathcal{N}(\mu, \sigma^2 I)$ doesn't respect the disk geometry:
- It assigns non-zero probability **outside** the disk ($\|x\| > 1$)
- It treats near-boundary points as "close together" when they are hyperbolically very far apart
- Its contours are Euclidean circles, not hyperbolic ones

### The Wrapped Normal Approach

We define a density that replaces Euclidean distance with Poincaré distance in the exponent:

$$p(x \mid \mu, \sigma) \propto \exp\!\left(-\frac{d_H(x, \mu)^2}{2\sigma^2}\right)$$

where $d_H(x, \mu)$ is the Poincaré disk distance. This is simpler than the true Riemannian normal distribution (which involves the heat kernel on hyperbolic space), but it captures the essential geometry: density contours follow **hyperbolic circles**, and the spread parameter $\sigma$ controls how concentrated the distribution is in hyperbolic distance.

The full density is:

$$p(x \mid \mu, \sigma) = \frac{1}{Z(\sigma)} \exp\!\left(-\frac{d_H(x, \mu)^2}{2\sigma^2}\right)$$

where $Z(\sigma)$ is a normalizing constant that we approximate numerically.

## 1. Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from itertools import combinations

np.random.seed(42)

cluster_colors = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12', '#9b59b6']

print("Setup complete!")

## 2. Generate Pre-Clustered Points in the Poincaré Disk

We reuse the Möbius addition approach from `hyperbolic_kmeans.ipynb`. Each cluster is generated by:
1. Choosing a seed center at some radius and angle inside the disk
2. Generating small Euclidean perturbations near the origin
3. Translating them to the cluster center via **Möbius addition** (the natural "shift" operation in hyperbolic space)

We use 5 clusters with **different spreads** -- some tight, some wide -- to test whether the EM algorithm can recover varying $\sigma$ values.

In [ ]:
def mobius_addition(a, b):
    """
    Mobius addition: a (+) b in the Poincare disk.
    This is the hyperbolic equivalent of vector addition.
    
    a (+) b = ((1 + 2<a,b> + |b|^2)a + (1 - |a|^2)b) / (1 + 2<a,b> + |a|^2|b|^2)
    """
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    
    a_dot_b = np.sum(a * b, axis=-1, keepdims=True)
    a_sq = np.sum(a**2, axis=-1, keepdims=True)
    b_sq = np.sum(b**2, axis=-1, keepdims=True)
    
    numerator = (1 + 2 * a_dot_b + b_sq) * a + (1 - a_sq) * b
    denominator = 1 + 2 * a_dot_b + a_sq * b_sq
    
    result = numerator / denominator
    
    # Ensure result stays inside the disk (numerical safety)
    norm = np.sqrt(np.sum(result**2, axis=-1, keepdims=True))
    max_norm = 0.999
    result = np.where(norm > max_norm, result * max_norm / norm, result)
    
    return result


def poincare_distance(u, v):
    """
    Hyperbolic distance in the Poincare disk model.
    d(u,v) = arcosh(1 + 2||u-v||^2 / ((1-||u||^2)(1-||v||^2)))
    """
    u = np.asarray(u, dtype=np.float64)
    v = np.asarray(v, dtype=np.float64)
    
    diff_sq = np.sum((u - v)**2, axis=-1)
    u_sq = np.sum(u**2, axis=-1)
    v_sq = np.sum(v**2, axis=-1)
    
    # Clip for numerical safety near the boundary
    denom = np.clip((1 - u_sq) * (1 - v_sq), 1e-15, None)
    
    arg = 1 + 2 * diff_sq / denom
    arg = np.clip(arg, 1.0, None)  # arcosh domain: [1, inf)
    
    return np.arccosh(arg)


def poincare_distance_matrix(points, centers):
    """Distance matrix between all points and all centers. Shape: (n_points, n_centers)."""
    n_p = len(points)
    n_c = len(centers)
    dist_matrix = np.zeros((n_p, n_c))
    for j in range(n_c):
        dist_matrix[:, j] = poincare_distance(points, centers[j])
    return dist_matrix


# Generate clustered data with VARYING spreads
def generate_hyperbolic_clusters(n_points=300, n_clusters=5, seed=42):
    """
    Generate pre-clustered points in the Poincare disk.
    Each cluster has a different spread to test whether EM can recover it.
    """
    rng = np.random.default_rng(seed)
    
    # Seed centers: (radius, angle, spread)
    center_configs = [
        (0.15, 0.0,   0.06),   # near center, tight
        (0.50, 2.0,   0.10),   # mid-depth, wider
        (0.75, 4.2,   0.05),   # deeper, tight
        (0.88, 1.2,   0.08),   # near boundary, medium
        (0.60, 5.0,   0.12),   # mid-depth, widest
    ]
    
    centers = []
    true_spreads = []
    for r, theta, spread in center_configs[:n_clusters]:
        centers.append(np.array([r * np.cos(theta), r * np.sin(theta)]))
        true_spreads.append(spread)
    centers = np.array(centers)
    
    points_per_cluster = n_points // n_clusters
    all_points = []
    all_labels = []
    
    for i in range(n_clusters):
        n = points_per_cluster if i < n_clusters - 1 else n_points - points_per_cluster * (n_clusters - 1)
        spread = true_spreads[i]
        
        # Generate small perturbations near the origin
        perturbations = rng.normal(0, spread, size=(n, 2))
        # Clip to stay well inside the disk
        pert_norms = np.sqrt(np.sum(perturbations**2, axis=1, keepdims=True))
        perturbations = np.where(pert_norms > 0.4, perturbations * 0.4 / pert_norms, perturbations)
        
        # Translate perturbations to the cluster center via Mobius addition
        cluster_points = mobius_addition(centers[i], perturbations)
        
        all_points.append(cluster_points)
        all_labels.append(np.full(n, i))
    
    return np.vstack(all_points), np.concatenate(all_labels), centers, np.array(true_spreads)


# Generate the dataset
points, true_labels, seed_centers, true_spreads = generate_hyperbolic_clusters(
    n_points=300, n_clusters=5
)

# Compute the true hyperbolic spreads (RMS Poincare distance from each center)
true_sigma_hyp = np.zeros(5)
for k in range(5):
    mask = true_labels == k
    dists = poincare_distance(points[mask], seed_centers[k])
    true_sigma_hyp[k] = np.sqrt(np.mean(dists**2))

print(f"Generated {len(points)} points in {len(np.unique(true_labels))} clusters")
print(f"Euclidean radii of seed centers: {[f'{np.linalg.norm(c):.2f}' for c in seed_centers]}")
print(f"Euclidean spreads used: {[f'{s:.2f}' for s in true_spreads]}")
print(f"True hyperbolic sigma (RMS d_H): {[f'{s:.3f}' for s in true_sigma_hyp]}")
print(f"All points inside unit disk: {np.all(np.linalg.norm(points, axis=1) < 1)}")

## 3. The Wrapped Normal Distribution on the Poincaré Disk

The density of a wrapped normal on the Poincaré disk:

$$p(x \mid \mu, \sigma) = \frac{1}{Z(\sigma)} \exp\!\left(-\frac{d_H(x, \mu)^2}{2\sigma^2}\right)$$

where:
- $d_H(x, \mu)$ is the Poincaré disk distance
- $\sigma > 0$ is the spread parameter
- $Z(\sigma)$ is the normalizing constant

### The Normalizing Constant

We **approximate** $Z(\sigma)$ numerically. For the E-step, only **relative** densities matter -- the normalization cancels in the ratio $r_{nk} = \pi_k p_k / \sum_j \pi_j p_j$ **only if all components share the same $\sigma$**. Since our clusters have different spreads, we need $Z(\sigma_k)$ for correctness.

The hyperbolic area element on the Poincaré disk is:

$$dA_{\text{hyp}} = \frac{4}{(1 - \|x\|^2)^2} \, dx_1 \, dx_2$$

So the normalizing constant (centered at the origin, which is equivalent to any center by isometry) is:

$$Z(\sigma) = \int_{\mathbb{D}} \exp\!\left(-\frac{d_H(x, 0)^2}{2\sigma^2}\right) \frac{4}{(1 - \|x\|^2)^2} \, dx_1 \, dx_2$$

We compute this by numerical integration over a grid inside the disk.

In [ ]:
def compute_normalizing_constant(sigma, grid_res=200):
    """
    Numerically approximate Z(sigma) for the wrapped normal on the Poincare disk.
    
    Uses a uniform grid in Euclidean coords, with the hyperbolic area element.
    By isometry invariance, Z(sigma) is the same regardless of the center mu.
    """
    # Grid over the disk
    x = np.linspace(-0.995, 0.995, grid_res)
    y = np.linspace(-0.995, 0.995, grid_res)
    dx = x[1] - x[0]
    dy = y[1] - y[0]
    xx, yy = np.meshgrid(x, y)
    grid = np.stack([xx.ravel(), yy.ravel()], axis=-1)
    
    r_sq = np.sum(grid**2, axis=1)
    inside = r_sq < 0.99  # stay away from boundary
    
    # Poincare distance from origin: d = 2 * arctanh(r)
    r = np.sqrt(r_sq[inside])
    d_H = 2 * np.arctanh(np.clip(r, 0, 0.999))
    
    # Wrapped normal kernel
    log_kernel = -d_H**2 / (2 * sigma**2)
    
    # Hyperbolic area element: 4 / (1 - r^2)^2
    area_element = 4.0 / (1 - r_sq[inside])**2
    
    # Numerical integral
    Z = np.sum(np.exp(log_kernel) * area_element) * dx * dy
    
    return Z


# Cache normalizing constants for a range of sigma values
sigma_grid = np.linspace(0.05, 3.0, 200)
Z_cache = {}
for s in sigma_grid:
    Z_cache[round(s, 4)] = compute_normalizing_constant(s)


def get_log_Z(sigma):
    """Get log Z(sigma) by interpolation from cache."""
    s_rounded = round(sigma, 4)
    if s_rounded in Z_cache:
        return np.log(Z_cache[s_rounded])
    # Linear interpolation
    idx = np.searchsorted(sigma_grid, sigma)
    idx = np.clip(idx, 1, len(sigma_grid) - 1)
    s_lo, s_hi = sigma_grid[idx - 1], sigma_grid[idx]
    Z_lo = Z_cache[round(s_lo, 4)]
    Z_hi = Z_cache[round(s_hi, 4)]
    t = (sigma - s_lo) / (s_hi - s_lo + 1e-15)
    Z_interp = Z_lo + t * (Z_hi - Z_lo)
    return np.log(max(Z_interp, 1e-30))


# Verify: Z should increase with sigma (wider distribution = more spread)
test_sigmas = [0.1, 0.3, 0.5, 1.0, 2.0]
print("Normalizing constants Z(sigma):")
for s in test_sigmas:
    Z = compute_normalizing_constant(s)
    print(f"  sigma = {s:.1f}  ->  Z = {Z:.4f}  (log Z = {np.log(Z):.4f})")

## 4. EM Algorithm

We now implement the full EM algorithm for a mixture of wrapped normals on the Poincaré disk.

### E-step

Compute responsibilities (soft assignments):

$$r_{nk} = \frac{\pi_k \, p(x_n \mid \mu_k, \sigma_k)}{\sum_{j=1}^{K} \pi_j \, p(x_n \mid \mu_j, \sigma_j)}$$

In log-space (using log-sum-exp for numerical stability):

$$\log r_{nk} = \log \pi_k - \log Z(\sigma_k) - \frac{d_H(x_n, \mu_k)^2}{2\sigma_k^2} - \log \sum_j \exp\!\left(\log \pi_j - \log Z(\sigma_j) - \frac{d_H(x_n, \mu_j)^2}{2\sigma_j^2}\right)$$

### M-step

**Mean $\mu_k$:** Use the **weighted Einstein midpoint** -- the same approach as K-means, but with responsibility weights instead of hard assignments.

**Spread $\sigma_k$:** Weighted RMS Poincaré distance:

$$\sigma_k = \sqrt{\frac{\sum_n r_{nk} \, d_H(x_n, \mu_k)^2}{N_k}}$$

where $N_k = \sum_n r_{nk}$ is the effective cluster size.

**Mixture weights:** $\pi_k = N_k / N$

In [ ]:
# --- Model helper functions ---

def poincare_to_klein(p):
    """Map from Poincare disk to Klein disk model."""
    p = np.asarray(p, dtype=np.float64)
    p_sq = np.sum(p**2, axis=-1, keepdims=True)
    return 2 * p / (1 + p_sq)


def klein_to_poincare(k):
    """Map from Klein disk to Poincare disk model."""
    k = np.asarray(k, dtype=np.float64)
    k_sq = np.sum(k**2, axis=-1, keepdims=True)
    denom = 1 + np.sqrt(np.clip(1 - k_sq, 1e-15, None))
    return k / denom


def weighted_einstein_midpoint(points, weights):
    """
    Compute the weighted Einstein midpoint (hyperbolic centroid).
    
    Same as einstein_midpoint but with responsibility weights:
    1. Map points from Poincare to Klein model
    2. Weighted average using Lorentz factors * responsibility weights
    3. Map result back to Poincare model
    """
    points = np.asarray(points, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    
    # Lorentz factors (gamma) for each point
    p_sq = np.sum(points**2, axis=-1)  # ||p||^2 in Poincare
    gamma = 1.0 / np.sqrt(np.clip(1 - p_sq, 1e-15, None))  # Lorentz factor
    
    # Map to Klein
    klein_points = poincare_to_klein(points)
    
    # Weighted average in Klein model
    combined_weights = gamma * weights  # shape: (n_points,)
    total_weight = np.sum(combined_weights)
    if total_weight < 1e-15:
        return np.array([0.0, 0.0])  # fallback to origin
    
    weighted_sum = np.sum(combined_weights[:, np.newaxis] * klein_points, axis=0)
    klein_mean = weighted_sum / total_weight
    
    # Ensure it stays inside the disk
    km_norm = np.linalg.norm(klein_mean)
    if km_norm >= 1.0:
        klein_mean = klein_mean * 0.999 / km_norm
    
    # Map back to Poincare
    return klein_to_poincare(klein_mean)


def logsumexp(a, axis=None):
    """Numerically stable log-sum-exp."""
    a_max = np.max(a, axis=axis, keepdims=True)
    result = a_max + np.log(np.sum(np.exp(a - a_max), axis=axis, keepdims=True))
    if axis is not None:
        result = np.squeeze(result, axis=axis)
    return result


print("Helper functions defined.")

# Quick test of weighted Einstein midpoint
test_pts = np.array([[0.3, 0.0], [-0.3, 0.0]])
test_w = np.array([1.0, 1.0])
mid = weighted_einstein_midpoint(test_pts, test_w)
print(f"Midpoint of (0.3,0) and (-0.3,0) with equal weights: ({mid[0]:.6f}, {mid[1]:.6f})")
print("  (Should be near the origin)")

In [ ]:
def wrapped_normal_em(points, K=5, max_iter=100, tol=1e-6, seed=123):
    """
    EM algorithm for a mixture of wrapped normals on the Poincare disk.
    
    Parameters:
        points: (N, 2) array of points inside the unit disk
        K: number of mixture components
        max_iter: maximum EM iterations
        tol: convergence threshold on log-likelihood change
        seed: random seed for initialization
    
    Returns:
        labels: hard assignments (argmax of responsibilities)
        responsibilities: (N, K) soft assignments
        means: (K, 2) cluster centers
        sigmas: (K,) spread parameters
        weights: (K,) mixture weights
        log_likelihoods: list of log-likelihood values per iteration
    """
    rng = np.random.default_rng(seed)
    N = len(points)
    
    # --- Initialization (K-means warm start) ---
    # Run a few iterations of hyperbolic K-means to get good initial means
    init_idx = rng.choice(N, size=K, replace=False)
    means = points[init_idx].copy()
    for _init_iter in range(20):
        dm = poincare_distance_matrix(points, means)
        init_labels = np.argmin(dm, axis=1)
        new_means = means.copy()
        for j in range(K):
            members = points[init_labels == j]
            if len(members) > 0:
                new_means[j] = weighted_einstein_midpoint(members, np.ones(len(members)))
        if np.max(np.linalg.norm(new_means - means, axis=1)) < 1e-6:
            break
        means = new_means
    
    # Initialize sigmas from K-means cluster spread
    dm = poincare_distance_matrix(points, means)
    init_labels = np.argmin(dm, axis=1)
    sigmas = np.zeros(K)
    for j in range(K):
        members_d = dm[init_labels == j, j]
        sigmas[j] = np.sqrt(np.mean(members_d**2)) if len(members_d) > 0 else 0.5
    sigmas = np.clip(sigmas, 0.05, 2.0)
    weights = np.full(K, 1.0 / K)  # uniform weights
    
    print(f"Initial means (Euclidean norms): {[f'{np.linalg.norm(m):.3f}' for m in means]}")
    print(f"Initial sigmas: {[f'{s:.3f}' for s in sigmas]}")
    
    log_likelihoods = []
    
    for iteration in range(max_iter):
        # === E-step ===
        # Compute log p(x_n | mu_k, sigma_k) for all n, k
        dist_matrix = poincare_distance_matrix(points, means)  # (N, K)
        
        # log-unnormalized density: -d_H^2 / (2*sigma_k^2)
        log_density = np.zeros((N, K))
        for k in range(K):
            log_Z_k = get_log_Z(np.clip(sigmas[k], 0.05, 3.0))
            log_density[:, k] = (np.log(weights[k] + 1e-30)
                                 - log_Z_k
                                 - dist_matrix[:, k]**2 / (2 * sigmas[k]**2))
        
        # Log-sum-exp for normalization
        log_sum = logsumexp(log_density, axis=1)  # (N,)
        log_resp = log_density - log_sum[:, np.newaxis]
        responsibilities = np.exp(log_resp)  # (N, K)
        
        # Clip for numerical safety
        responsibilities = np.clip(responsibilities, 1e-15, None)
        responsibilities /= responsibilities.sum(axis=1, keepdims=True)
        
        # Log-likelihood
        ll = np.sum(log_sum)
        log_likelihoods.append(ll)
        
        # === M-step ===
        N_k = responsibilities.sum(axis=0)  # (K,) effective cluster sizes
        
        new_means = np.zeros_like(means)
        new_sigmas = np.zeros(K)
        new_weights = N_k / N
        
        for k in range(K):
            # Weighted Einstein midpoint
            new_means[k] = weighted_einstein_midpoint(points, responsibilities[:, k])
            
            # Weighted RMS Poincare distance
            dists_k = poincare_distance(points, new_means[k])
            new_sigmas[k] = np.sqrt(np.sum(responsibilities[:, k] * dists_k**2) / N_k[k])
            # Clamp sigma to reasonable range
            new_sigmas[k] = np.clip(new_sigmas[k], 0.05, 3.0)
        
        # Check convergence
        if iteration > 0:
            ll_change = abs(log_likelihoods[-1] - log_likelihoods[-2])
            if ll_change < tol:
                print(f"Converged at iteration {iteration + 1} (delta LL = {ll_change:.2e})")
                means = new_means
                sigmas = new_sigmas
                weights = new_weights
                break
        
        # Print progress every 10 iterations
        if (iteration + 1) % 10 == 0 or iteration == 0:
            print(f"  Iter {iteration + 1:3d}: LL = {ll:.2f}, "
                  f"sigmas = {[f'{s:.3f}' for s in new_sigmas]}, "
                  f"max|dmu| = {np.max(poincare_distance(means, new_means)):.6f}")
        
        means = new_means
        sigmas = new_sigmas
        weights = new_weights
    else:
        print(f"Reached max iterations ({max_iter})")
    
    labels = np.argmax(responsibilities, axis=1)
    
    print(f"\nFinal cluster sizes: {[np.sum(labels == k) for k in range(K)]}")
    print(f"Final sigmas: {[f'{s:.4f}' for s in sigmas]}")
    print(f"Final weights: {[f'{w:.4f}' for w in weights]}")
    
    return labels, responsibilities, means, sigmas, weights, log_likelihoods


# Run EM
K = 5
em_labels, em_resp, em_means, em_sigmas, em_weights, em_ll = wrapped_normal_em(
    points, K=K, max_iter=200, tol=1e-6, seed=123
)

## 5. Visualizations

### 5a. Poincaré Disk with Clustered Points and Density Contours

We show the wrapped normal mixture on the Poincaré disk with:
- Background: soft Voronoi shading by posterior probability
- Points colored by hard assignment
- Centroids as black stars
- Concentric hyperbolic distance circles from the origin ($d = 1, 2, 3$)
- Per-component density contours

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

# Draw the unit disk boundary
boundary = plt.Circle((0, 0), 1, fill=False, color='black', linewidth=2)
ax.add_patch(boundary)

# --- Background: soft posterior shading ---
grid_res = 300
gx = np.linspace(-1, 1, grid_res)
gy = np.linspace(-1, 1, grid_res)
gx_mesh, gy_mesh = np.meshgrid(gx, gy)
grid_pts = np.stack([gx_mesh.ravel(), gy_mesh.ravel()], axis=-1)

inside_mask = np.sum(grid_pts**2, axis=1) < 0.98

# Compute posterior at each grid point
grid_log_density = np.full((len(grid_pts), K), -np.inf)
if np.any(inside_mask):
    grid_dists = poincare_distance_matrix(grid_pts[inside_mask], em_means)
    for k in range(K):
        log_Z_k = get_log_Z(np.clip(em_sigmas[k], 0.05, 3.0))
        grid_log_density[inside_mask, k] = (np.log(em_weights[k] + 1e-30)
                                            - log_Z_k
                                            - grid_dists[:, k]**2 / (2 * em_sigmas[k]**2))

grid_log_sum = logsumexp(grid_log_density[inside_mask], axis=1)
grid_resp_inside = np.exp(grid_log_density[inside_mask] - grid_log_sum[:, np.newaxis])
grid_labels_bg = np.argmax(grid_log_density, axis=1)

grid_labels_img = grid_labels_bg.reshape(grid_res, grid_res)

# Create RGBA image for background
bg_rgba = np.ones((grid_res, grid_res, 4))
for i in range(K):
    mask_2d = grid_labels_img == i
    rgba = mcolors.to_rgba(cluster_colors[i], alpha=0.12)
    bg_rgba[mask_2d] = rgba

outside_2d = ~inside_mask.reshape(grid_res, grid_res)
bg_rgba[outside_2d] = [1, 1, 1, 1]

ax.imshow(bg_rgba, extent=[-1, 1, -1, 1], origin='lower', aspect='equal')

# --- Per-component density contours ---
for k in range(K):
    # Compute density for this component on grid
    comp_density = np.full(len(grid_pts), 0.0)
    if np.any(inside_mask):
        d_k = poincare_distance(grid_pts[inside_mask], em_means[k])
        comp_density[inside_mask] = np.exp(-d_k**2 / (2 * em_sigmas[k]**2))
    
    comp_img = comp_density.reshape(grid_res, grid_res)
    levels = [0.1, 0.3, 0.6]
    ax.contour(gx_mesh, gy_mesh, comp_img, levels=levels,
               colors=[cluster_colors[k]], alpha=0.5, linewidths=1.0, linestyles='--')

# --- Points colored by assignment ---
for i in range(K):
    mask = em_labels == i
    ax.scatter(points[mask, 0], points[mask, 1],
               c=cluster_colors[i], s=30, alpha=0.8, edgecolors='white',
               linewidths=0.4, label=f'Cluster {i+1} ($\\sigma$={em_sigmas[i]:.2f})', zorder=5)

# --- Centroids ---
ax.scatter(em_means[:, 0], em_means[:, 1],
           c='black', marker='*', s=350, edgecolors='white',
           linewidths=1.5, zorder=10, label='EM Centroids')

# --- Concentric hyperbolic distance circles from origin ---
for r_hyp in [1, 2, 3]:
    r_euc = np.tanh(r_hyp / 2)
    circle = plt.Circle((0, 0), r_euc, fill=False, color='gray',
                        linewidth=0.5, linestyle=':', alpha=0.5)
    ax.add_patch(circle)
    ax.annotate(f'd={r_hyp}', xy=(r_euc * 0.707, r_euc * 0.707),
                fontsize=8, color='gray', alpha=0.7)

ax.set_xlim(-1.15, 1.15)
ax.set_ylim(-1.15, 1.15)
ax.set_aspect('equal')
ax.legend(loc='upper right', fontsize=9)
ax.set_title('Wrapped Normal Mixture on the Poincar\u00e9 Disk\n'
             '(dashed = density contours, dotted circles = hyperbolic distance from origin)',
             fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.15)

plt.tight_layout()
plt.show()

### 5b. Soft Assignment Visualization

Unlike K-means, the wrapped normal mixture gives each point a **probability** of belonging to each cluster. Points near cluster boundaries have low confidence (max responsibility near $1/K$), while core points have high confidence (max responsibility near 1).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# --- Left: Points with opacity = max responsibility ---
ax = axes[0]
boundary = plt.Circle((0, 0), 1, fill=False, color='black', linewidth=2)
ax.add_patch(boundary)

max_resp = np.max(em_resp, axis=1)  # confidence per point

for i in range(K):
    mask = em_labels == i
    pts_i = points[mask]
    conf_i = max_resp[mask]
    for j in range(len(pts_i)):
        ax.scatter(pts_i[j, 0], pts_i[j, 1],
                   c=cluster_colors[i], s=30, alpha=float(conf_i[j]),
                   edgecolors='none', zorder=5)

ax.scatter(em_means[:, 0], em_means[:, 1],
           c='black', marker='*', s=300, edgecolors='white',
           linewidths=1.5, zorder=10)

for r_hyp in [1, 2, 3]:
    r_euc = np.tanh(r_hyp / 2)
    circle = plt.Circle((0, 0), r_euc, fill=False, color='gray',
                        linewidth=0.5, linestyle=':', alpha=0.4)
    ax.add_patch(circle)

ax.set_xlim(-1.15, 1.15)
ax.set_ylim(-1.15, 1.15)
ax.set_aspect('equal')
ax.set_title('Soft Assignments (opacity = confidence)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.15)

# --- Right: Histogram of confidence ---
ax = axes[1]
ax.hist(max_resp, bins=30, color='#3498db', edgecolor='white', alpha=0.8)
ax.axvline(1.0 / K, color='#e74c3c', linestyle='--', linewidth=2,
           label=f'Uniform = {1.0/K:.2f}')
ax.axvline(np.mean(max_resp), color='#2ecc71', linestyle='--', linewidth=2,
           label=f'Mean = {np.mean(max_resp):.3f}')
ax.set_xlabel('Max Responsibility (Confidence)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Distribution of Assignment Confidence', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

n_high = np.sum(max_resp > 0.9)
n_low = np.sum(max_resp < 0.5)
ax.annotate(f'{n_high} points with > 90% confidence',
            xy=(0.95, ax.get_ylim()[1] * 0.85), fontsize=10, ha='right',
            color='#2c3e50')
ax.annotate(f'{n_low} points with < 50% confidence',
            xy=(0.45, ax.get_ylim()[1] * 0.75), fontsize=10, ha='right',
            color='#e74c3c')

plt.tight_layout()
plt.show()

### 5c. Convergence Plot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

iterations = range(1, len(em_ll) + 1)

ax.plot(iterations, em_ll, 'o-', color='#2c3e50', markersize=5,
        linewidth=2, markerfacecolor='#9b59b6', markeredgecolor='white', markeredgewidth=1.2)

ax.fill_between(iterations, em_ll, alpha=0.1, color='#9b59b6')

ax.set_xlabel('EM Iteration', fontsize=12)
ax.set_ylabel('Log-Likelihood', fontsize=12)
ax.set_title('Wrapped Normal Mixture EM Convergence', fontsize=14, fontweight='bold')

ax.annotate(f'Start: {em_ll[0]:.1f}', xy=(1, em_ll[0]),
            xytext=(max(2, len(em_ll) * 0.15), em_ll[0] + (em_ll[-1] - em_ll[0]) * 0.1),
            arrowprops=dict(arrowstyle='->', color='gray'), fontsize=10, color='gray')
ax.annotate(f'Final: {em_ll[-1]:.1f}', xy=(len(em_ll), em_ll[-1]),
            xytext=(max(1, len(em_ll) * 0.7), em_ll[-1] - (em_ll[-1] - em_ll[0]) * 0.15),
            arrowprops=dict(arrowstyle='->', color='gray'), fontsize=10, color='gray')

ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5d. Spread Comparison: True vs Estimated $\sigma$

One key advantage of the mixture model over K-means is that it estimates per-cluster **spread** $\sigma_k$. Let's compare the estimated values against the true hyperbolic spreads.

In [ ]:
# Match estimated clusters to true clusters by nearest centroid
# (clusters may be in a different order)
dist_ct = poincare_distance_matrix(em_means, seed_centers)  # (K, K)
# Greedy matching
matched_true_idx = []
used = set()
for k in range(K):
    # Find nearest true center not yet used
    order = np.argsort(dist_ct[k])
    for idx in order:
        if idx not in used:
            matched_true_idx.append(idx)
            used.add(idx)
            break

matched_true_sigma = true_sigma_hyp[matched_true_idx]

print("Cluster matching (EM -> True):")
for k in range(K):
    d = dist_ct[k, matched_true_idx[k]]
    print(f"  EM cluster {k} -> True cluster {matched_true_idx[k]}  "
          f"(centroid dist = {d:.4f}, sigma: est={em_sigmas[k]:.4f} vs true={matched_true_sigma[k]:.4f})")

fig, ax = plt.subplots(figsize=(9, 5))

x_pos = np.arange(K)
bar_width = 0.35

bars_true = ax.bar(x_pos - bar_width/2, matched_true_sigma, bar_width,
                   label='True $\\sigma$ (RMS $d_H$)', color='#3498db', alpha=0.8,
                   edgecolor='white', linewidth=1.5)
bars_est = ax.bar(x_pos + bar_width/2, em_sigmas, bar_width,
                  label='Estimated $\\sigma$ (EM)', color='#e74c3c', alpha=0.8,
                  edgecolor='white', linewidth=1.5)

# Value labels
for bar in bars_true:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.3f}',
            ha='center', va='bottom', fontsize=9, color='#3498db')
for bar in bars_est:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.3f}',
            ha='center', va='bottom', fontsize=9, color='#e74c3c')

ax.set_xlabel('Cluster', fontsize=12)
ax.set_ylabel('Spread $\\sigma$', fontsize=12)
ax.set_title('True vs Estimated Spread per Cluster', fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'Cluster {k+1}' for k in range(K)])
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 6. Hyperbolic K-Means vs Wrapped Normal Mixture

Let's run hyperbolic K-means on the same data and compare the two approaches side by side.

In [ ]:
def einstein_midpoint(points):
    """
    Compute the Einstein midpoint (hyperbolic centroid) of a set of points.
    1. Map points from Poincare to Klein model
    2. Lorentz-factor-weighted average in Klein model
    3. Map result back to Poincare model
    """
    points = np.asarray(points, dtype=np.float64)
    p_sq = np.sum(points**2, axis=-1)
    gamma = 1.0 / np.sqrt(np.clip(1 - p_sq, 1e-15, None))
    klein_points = poincare_to_klein(points)
    weights = gamma
    weighted_sum = np.sum(weights[:, np.newaxis] * klein_points, axis=0)
    klein_mean = weighted_sum / np.sum(weights)
    km_norm = np.linalg.norm(klein_mean)
    if km_norm >= 1.0:
        klein_mean = klein_mean * 0.999 / km_norm
    return klein_to_poincare(klein_mean)


def hyperbolic_kmeans(points, k=5, max_iter=100, tol=1e-6, seed=123):
    """
    K-means in the Poincare disk using hyperbolic distance
    and Einstein midpoint centroids.
    """
    rng = np.random.default_rng(seed)
    n = len(points)
    
    init_idx = rng.choice(n, size=k, replace=False)
    centroids = points[init_idx].copy()
    
    history = []
    labels = np.zeros(n, dtype=int)
    
    for iteration in range(max_iter):
        dist_matrix = poincare_distance_matrix(points, centroids)
        labels = np.argmin(dist_matrix, axis=1)
        
        total_dist = np.sum(dist_matrix[np.arange(n), labels])
        history.append(total_dist)
        
        new_centroids = np.zeros_like(centroids)
        for j in range(k):
            members = points[labels == j]
            if len(members) == 0:
                new_centroids[j] = points[rng.choice(n)]
            else:
                new_centroids[j] = einstein_midpoint(members)
        
        shift = poincare_distance(centroids, new_centroids)
        centroids = new_centroids
        
        if np.max(shift) < tol:
            print(f"Hyperbolic K-means converged at iteration {iteration + 1}")
            break
    else:
        print(f"Reached max iterations ({max_iter})")
    
    return labels, centroids, history


# Run hyperbolic K-means
hyp_labels, hyp_centroids, hyp_history = hyperbolic_kmeans(
    points, k=K, max_iter=100, seed=123
)

print(f"\nK-means cluster sizes: {[np.sum(hyp_labels == k) for k in range(K)]}")
print(f"EM cluster sizes:      {[np.sum(em_labels == k) for k in range(K)]}")

In [ ]:
# --- Side-by-side comparison ---
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

titles = ['Hyperbolic K-Means (Hard)', 'Wrapped Normal Mixture (Soft EM)']
all_labels_list = [hyp_labels, em_labels]
all_centroids_list = [hyp_centroids, em_means]

for idx, (ax, title, lbls, cents) in enumerate(
    zip(axes, titles, all_labels_list, all_centroids_list)
):
    # Disk boundary
    boundary = plt.Circle((0, 0), 1, fill=False, color='black', linewidth=2)
    ax.add_patch(boundary)
    
    # Background Voronoi shading
    grid_dists_bg = poincare_distance_matrix(grid_pts[inside_mask], cents)
    grid_labels_local = np.full(len(grid_pts), -1)
    grid_labels_local[inside_mask] = np.argmin(grid_dists_bg, axis=1)
    
    grid_img = grid_labels_local.reshape(grid_res, grid_res)
    bg = np.ones((grid_res, grid_res, 4))
    for i in range(K):
        mask_2d = grid_img == i
        bg[mask_2d] = mcolors.to_rgba(cluster_colors[i], alpha=0.12)
    bg[grid_img == -1] = [1, 1, 1, 1]
    ax.imshow(bg, extent=[-1, 1, -1, 1], origin='lower', aspect='equal')
    
    # Points
    for i in range(K):
        mask = lbls == i
        ax.scatter(points[mask, 0], points[mask, 1],
                   c=cluster_colors[i], s=25, alpha=0.8, edgecolors='white',
                   linewidths=0.3, label=f'Cluster {i+1}', zorder=5)
    
    # Centroids
    ax.scatter(cents[:, 0], cents[:, 1],
               c='black', marker='*', s=300, edgecolors='white',
               linewidths=1.5, zorder=10, label='Centroids')
    
    # Density contours for EM
    if idx == 1:
        for k in range(K):
            comp_density = np.full(len(grid_pts), 0.0)
            d_k = poincare_distance(grid_pts[inside_mask], em_means[k])
            comp_density[inside_mask] = np.exp(-d_k**2 / (2 * em_sigmas[k]**2))
            comp_img = comp_density.reshape(grid_res, grid_res)
            ax.contour(gx_mesh, gy_mesh, comp_img, levels=[0.1, 0.3, 0.6],
                       colors=[cluster_colors[k]], alpha=0.4, linewidths=0.8, linestyles='--')
    
    # Hyperbolic distance rings
    for r_hyp in [1, 2, 3]:
        r_euc = np.tanh(r_hyp / 2)
        circle = plt.Circle((0, 0), r_euc, fill=False, color='gray',
                            linewidth=0.5, linestyle=':', alpha=0.4)
        ax.add_patch(circle)
    
    ax.set_xlim(-1.15, 1.15)
    ax.set_ylim(-1.15, 1.15)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.15)

plt.suptitle('Hard vs Soft Clustering on the Poincar\u00e9 Disk',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Where do K-means and EM disagree? ---
disagree = hyp_labels != em_labels
# Note: labels may be permuted, so we need to find the best matching
# Use greedy matching based on centroid proximity
dist_km_em = poincare_distance_matrix(hyp_centroids, em_means)  # (K, K)
km_to_em = []
used_em = set()
for k in range(K):
    order = np.argsort(dist_km_em[k])
    for idx in order:
        if idx not in used_em:
            km_to_em.append(idx)
            used_em.add(idx)
            break

# Remap K-means labels to match EM labels
hyp_labels_remapped = np.array([km_to_em[l] for l in hyp_labels])
disagree = hyp_labels_remapped != em_labels
n_disagree = np.sum(disagree)

print(f"Points where K-means and EM disagree: {n_disagree} / {len(points)} ({100*n_disagree/len(points):.1f}%)")

# Rand Index
def rand_index(labels_a, labels_b):
    """Compute the Rand Index between two clusterings."""
    n = len(labels_a)
    agree = 0
    total = 0
    for i, j in combinations(range(n), 2):
        same_a = labels_a[i] == labels_a[j]
        same_b = labels_b[i] == labels_b[j]
        if same_a == same_b:
            agree += 1
        total += 1
    return agree / total if total > 0 else 1.0


ri_km_true = rand_index(hyp_labels_remapped, true_labels)
ri_em_true = rand_index(em_labels, true_labels)
ri_km_em = rand_index(hyp_labels_remapped, em_labels)

print(f"\nRand Index (agreement with ground truth):")
print(f"  Hyperbolic K-Means vs True: {ri_km_true:.4f}")
print(f"  Wrapped Normal EM vs True:  {ri_em_true:.4f}")
print(f"  K-Means vs EM:              {ri_km_em:.4f}")

In [ ]:
# --- Rand Index bar chart ---
fig, ax = plt.subplots(figsize=(9, 5))

methods = ['Hyp. K-Means\nvs True', 'Wrapped Normal EM\nvs True', 'K-Means\nvs EM']
ri_values = [ri_km_true, ri_em_true, ri_km_em]
bar_colors = ['#3498db', '#e74c3c', '#9b59b6']

bars = ax.barh(methods, ri_values, color=bar_colors, alpha=0.85,
               edgecolor='white', linewidth=2, height=0.5)

for bar, val in zip(bars, ri_values):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=12, fontweight='bold')

ax.set_xlabel('Rand Index (vs Ground Truth)', fontsize=12)
ax.set_title('Clustering Agreement: K-Means vs Wrapped Normal EM',
             fontsize=14, fontweight='bold')
ax.set_xlim(0, 1.08)
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## Choosing K — PVE & Silhouette Analysis

How do we know K = 5 is the right number of clusters? We use two metrics:

**PVE (Proportion of Variance Explained)**

$$\text{PVE}(K) = 1 - \frac{\text{TWCSS}_K}{\text{TWCSS}_1}$$

where $\text{TWCSS}_K$ is the **total within-cluster sum of squared distances** for K clusters and $\text{TWCSS}_1$ is the single-cluster baseline (all points assigned to one cluster). PVE ranges from 0 to 1; an "elbow" in the curve marks a natural K choice.

**Silhouette Score**

For each point $i$:

$$s(i) = \frac{b(i) - a(i)}{\max(a(i),\, b(i))} \in [-1, +1]$$

where $a(i)$ = mean distance to other points in the **same** cluster (cohesion), and $b(i)$ = mean distance to points in the **nearest other** cluster (separation). Computed from the precomputed pairwise distance matrix so it automatically respects this notebook's metric.

In [ ]:
def silhouette_from_matrix(D, labels):
    """
    Mean silhouette score from a precomputed N×N distance matrix.
    s(i) = (b(i) - a(i)) / max(a(i), b(i))
    """
    n = len(labels)
    unique_clusters = np.unique(labels)
    if len(unique_clusters) <= 1:
        return 0.0
    scores = np.zeros(n)
    for i in range(n):
        ci = labels[i]
        same_mask = (labels == ci).copy()
        same_mask[i] = False
        a_i = float(np.mean(D[i, same_mask])) if np.any(same_mask) else 0.0
        b_i = np.inf
        for cj in unique_clusters:
            if cj == ci:
                continue
            other_mask = (labels == cj)
            if np.any(other_mask):
                b_i = min(b_i, float(np.mean(D[i, other_mask])))
        denom = max(a_i, b_i)
        scores[i] = (b_i - a_i) / denom if denom > 1e-15 else 0.0
    return float(np.mean(scores))

In [ ]:
# For the wrapped-normal mixture, use hard assignments (em_labels = argmax of responsibilities)
# and Poincaré distance for TWCSS and silhouette.
# poincare_distance, poincare_distance_matrix, einstein_midpoint are all already defined above.

# Build full N×N Poincaré distance matrix
n_pts = len(points)
print("Building pairwise Poincaré distance matrix...")
D_full = poincare_distance_matrix(points, points)  # N×N

# TWCSS baseline: one cluster, centred on Einstein midpoint of all points
overall_center = einstein_midpoint(points)
TWCSS_1 = float(np.sum(poincare_distance(points, overall_center) ** 2))

K_range = range(2, 11)
pve_vals = []
sil_vals = []

print(f"{'K':>4} {'TWCSS':>10} {'PVE':>8} {'Silhouette':>12}")
print("-" * 38)
for k in K_range:
    lbls_k, _, means_k, _, _, _ = wrapped_normal_em(points, K=k, max_iter=100, seed=123)
    dists_k = poincare_distance(points, means_k[lbls_k])
    twcss_k = float(np.sum(dists_k ** 2))
    pve = 1 - twcss_k / TWCSS_1
    sil = silhouette_from_matrix(D_full, lbls_k)
    pve_vals.append(pve)
    sil_vals.append(sil)
    print(f"{k:>4} {twcss_k:>10.4f} {pve:>8.4f} {sil:>12.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
K_chosen = 5
K_vals = list(K_range)

ax = axes[0]
ax.plot(K_vals, pve_vals, 'o-', color='#2c3e50', linewidth=2, markersize=7,
        markerfacecolor='#3498db', markeredgecolor='white', markeredgewidth=1.5)
ax.axvline(K_chosen, color='#e74c3c', linestyle='--', linewidth=1.8,
           label=f'Chosen K = {K_chosen}')
ax.fill_between(K_vals, pve_vals, alpha=0.08, color='#3498db')
ax.set_xlabel('Number of Clusters K', fontsize=12)
ax.set_ylabel('PVE', fontsize=12)
ax.set_title('PVE Elbow Curve', fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.set_xticks(K_vals); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(K_vals, sil_vals, 's-', color='#2c3e50', linewidth=2, markersize=7,
        markerfacecolor='#9b59b6', markeredgecolor='white', markeredgewidth=1.5)
best_k_sil = K_vals[int(np.argmax(sil_vals))]
ax.axvline(best_k_sil, color='#9b59b6', linestyle='--', linewidth=1.8,
           label=f'Best silhouette K = {best_k_sil}')
ax.axvline(K_chosen, color='#e74c3c', linestyle=':', linewidth=1.8,
           label=f'Chosen K = {K_chosen}')
ax.fill_between(K_vals, sil_vals, alpha=0.08, color='#9b59b6')
ax.set_xlabel('Number of Clusters K', fontsize=12)
ax.set_ylabel('Mean Silhouette Score', fontsize=12)
ax.set_title('Silhouette Score vs K', fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.set_xticks(K_vals); ax.grid(True, alpha=0.3)

plt.suptitle('Choosing K: PVE & Silhouette Validation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nK=2..10 summary:")
print(f"{'K':>4} {'PVE':>8} {'Silhouette':>12}")
print("-" * 27)
for k, pve, sil in zip(K_vals, pve_vals, sil_vals):
    marker = " ◄ chosen" if k == K_chosen else ""
    print(f"{k:>4} {pve:>8.4f} {sil:>12.4f}{marker}")

## 7. Summary

| Aspect | Hyperbolic K-Means | Wrapped Normal Mixture (EM) |
|--------|-------------------|-----------------------------|
| **Assignment** | Hard (each point to exactly one cluster) | Soft (probability per cluster) |
| **Cluster shape** | Implicit Voronoi cells in Poincaré distance | Wrapped normal density contours (hyperbolic circles) |
| **Parameters** | Only centroids $\mu_k$ | Centroids $\mu_k$, spreads $\sigma_k$, weights $\pi_k$ |
| **Handles varying spread** | No -- all clusters treated equally | Yes -- each cluster has its own $\sigma_k$ |
| **Boundary type** | Hard Voronoi boundaries | Soft probabilistic boundaries |
| **Best for** | Fast exploratory clustering, well-separated clusters | Overlapping clusters, uncertainty quantification, varying cluster sizes |

### Key Takeaways

1. **Geometry matters in the exponent:** Replacing Euclidean distance with Poincaré distance in the Gaussian exponent gives density contours that follow hyperbolic circles -- they look like Euclidean circles near the disk center but become increasingly compressed near the boundary, matching the true geometry.

2. **Soft assignments provide uncertainty:** Near cluster boundaries or in overlap regions, the mixture model gives calibrated probabilities rather than forced hard decisions. This is especially valuable in hyperbolic space where the metric distortion can make boundary points ambiguous.

3. **Per-cluster spread recovery:** The EM algorithm recovers per-cluster spread parameters $\sigma_k$, capturing that some clusters are tighter than others. K-means has no notion of per-cluster scale.

4. **The normalizing constant is subtle:** Unlike Euclidean GMMs where $Z(\sigma)$ has a closed form, the wrapped normal on the Poincaré disk requires numerical approximation. For the E-step ratios this mostly cancels, but including it is important when $\sigma$ values differ substantially across components.

### Next: Toroidal -- von Mises Mixture on the Torus